# Spider brain driving Jumper (Colab, GPU)
Runs Jumper's own simulator with the **synthetic jumping-spider brain** holding the gamepad. **Untested end to end** — if a cell fails, copy its error.
Runtime → Change runtime type → **T4 GPU**, then Run all. Cells 1–5 are setup (~10 min); cell 6 is a no-brain baseline that proves Jumper itself works; cell 7 is the brain run.

In [ ]:
# 1. GPU check
!nvidia-smi -L
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import sys; print(sys.version)

In [ ]:
# 2. Jumper, pinned to the commit this was written against (third-party code: Apache-2.0)
JUMPER = '/content/jumper'
REV = '78966f45de6f2b6de3f7cea017f32a39f8b993e6'
import os
if not os.path.exists(JUMPER):
    !git clone https://github.com/KingKongRobotics/jumper.git {JUMPER}
!cd {JUMPER} && git checkout -q {REV} && git log -1 --format='%h %s'

In [ ]:
# 3. isolated venv (built with uv: Colab's `python -m venv` makes a venv WITHOUT pip) + torch + Jumper
# Do NOT pip install mjlab / rsl-rl-lib: Jumper vendors them.
import subprocess, sys, os
cap = float(subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split()[0])
CUDA = 'cu128' if cap >= 8.9 else 'cu126'     # Jumper's COLAB.md: 12.6 for T4, 12.8 for newer
print('compute cap', cap, '->', CUDA)
PY = f'{JUMPER}/.venv/bin/python'
!pip install -q uv
!rm -rf {JUMPER}/.venv
!uv venv --python {sys.executable} {JUMPER}/.venv
!uv pip install --python {PY} torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/{CUDA} 2>&1 | tail -5
!cd {JUMPER} && uv pip install --python {PY} -e . 2>&1 | tail -5

In [ ]:
# 4. verify Jumper's stack (expect cuda:0, and mjlab paths inside /content/jumper/rl)
!{PY} -c "import warp as wp; wp.init(); print(wp.get_devices())"
!{PY} -c "import mjlab, rsl_rl, os; print(os.path.dirname(mjlab.__file__)); print(os.path.dirname(rsl_rl.__file__))"
!{PY} -c "import onnxruntime; print('onnxruntime', onnxruntime.__version__)" 

In [ ]:
# 5. this repo, installed INTO Jumper's venv (numpy/scipy/pandas/pyarrow/pyyaml/pytest)
SB = '/content/spiderbrain-moveset'
if not os.path.exists(SB):
    !git clone https://github.com/adhritpradeepashoka/spiderbrain-moveset.git {SB}
!cd {SB} && git pull -q
!uv pip install --python {PY} -e "{SB}[dev]" 2>&1 | tail -3
!cd {SB} && {PY} -m pytest -q -p no:warnings 2>&1 | tail -3

In [ ]:
# 6. BASELINE: Jumper's prebuilt app with NO brain (keyboard/pad idle). If this fails, the problem is Jumper/Colab, not the spider brain.
!cd {JUMPER} && MUJOCO_GL=egl {PY} scripts/play.py --app out/bundle_example/jumper.app --headless --steps 300 --video /content/baseline.mp4
from IPython.display import Video
Video('/content/baseline.mp4', embed=True, width=480)

In [ ]:
# 7a. dry run of the brain launcher: resolves the app + scene without simulating
!cd {SB} && MUJOCO_GL=egl {PY} scripts/play_with_brain.py --jumper {JUMPER} --seed 0 --prey-entity ball -- --app out/bundle_example/jumper.app --scene football --headless --dry-run

In [ ]:
# 7b. THE RUN (~30 s of robot time: Jumper steps at ~200 Hz, so 6000 steps; use 12000 for a minute): spider brain (seed 0) steers toward a wandering ball and pounces. Fixed wide side camera so you can see the whole hunt.
# --video-azimuth 90 = side view (try 270 for the other side). Change --seed to meet a different random brain.
!cd {SB} && MUJOCO_GL=egl {PY} scripts/play_with_brain.py --jumper {JUMPER} --seed 0 --prey-entity ball --log /content/brain_log.csv -- --app out/bundle_example/jumper.app --scene football --headless --steps 6000 --video /content/brain_clip.mp4 --no-video-follow --video-lookat-height 0.1 --video-distance 3.5 --video-azimuth 90 2>&1 | grep -E "spiderbrain|Error|Traceback|error" | tail -45
from IPython.display import Video
Video('/content/brain_clip.mp4', embed=True, width=640)

In [ ]:
# 7c. What did the brain actually do? Top-down path of robot and prey, with commands over time.
import pandas as pd, matplotlib.pyplot as plt
d = pd.read_csv('/content/brain_log.csv')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(d.prey_x, d.prey_y, 'o-', ms=2, c='tab:orange', label='prey (ball)')
ax[0].plot(d.robot_x, d.robot_y, 'o-', ms=2, c='tab:red', label='robot')
g = d[d.pad_A == 1]
ax[0].scatter(g.robot_x, g.robot_y, c='k', s=60, marker='*', label='jump pressed (A)', zorder=5)
ax[0].set_aspect('equal'); ax[0].legend(); ax[0].set_title('top-down path (m)')
ax[1].plot(d.t_s, d.dist_m); ax[1].axhspan(0.05, 0.40, color='g', alpha=.15); ax[1].set_title('distance to prey (green = pounce window)')
ax[2].plot(d.t_s, d.turn_cmd, label='turn (+ = left)'); ax[2].plot(d.t_s, d.fwd_cmd, label='forward'); ax[2].plot(d.t_s, d.bearing_deg/60, label='prey bearing /60deg', alpha=.5); ax[2].legend(); ax[2].set_title('brain commands')
plt.tight_layout(); plt.show()
print('min distance %.2f m; ticks with jump pressed: %d; max |forward cmd| %.2f' % (d.dist_m.min(), (d.pad_A==1).sum(), d.fwd_cmd.abs().max()))

In [ ]:
# 8. download the clip (post it yourself; see docs/contest-checklist.md)
from google.colab import files
files.download('/content/brain_clip.mp4')

### If something fails
| Symptom | Likely cause / next step |
|---|---|
| cell 3 pip error | Python/CUDA mismatch; paste the last 30 lines |
| cell 6 fails | Jumper itself (EGL/GPU). Read its `docs/COLAB.md`; this is not the brain |
| 7a `KeyError` on `ball` | scene entity name differs; drop `--prey-entity` to use a fixed virtual prey (invisible) |
| 7b robot ignores the brain | check `deploy/manifests.json` bindings; look at the `[spiderbrain]` lines: is `gate=open` ever? |
| robot never pounces | prey never inside the 5–40 cm window while aligned; widen `pounce_gate` in `configs/movesets.yaml` |